In [ ]:
# Import libraries and load datasets
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer

# Load the full dataset (this grabs all splits)
ds_dict = load_dataset("SALT-NLP/CultureBank")
# Combine the 'tiktok' and 'reddit' splits into one main dataset
ds_cb = concatenate_datasets([ds_dict['tiktok'], ds_dict['reddit']])

# SNLI does have a standard 'train' split, so this stays the same
ds_snli = load_dataset("snli", split="train")

print(f"CultureBank initial size: {len(ds_cb)}")
print(f"SNLI initial size: {len(ds_snli)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/186 [00:00<?, ?B/s]

tiktok/culturebank_tiktok.csv:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

reddit/culturebank_reddit.csv:   0%|          | 0.00/20.7M [00:00<?, ?B/s]

Generating tiktok split:   0%|          | 0/11754 [00:00<?, ? examples/s]

...

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/19.6M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/550152 [00:00<?, ? examples/s]

KeyboardInterrupt: 

Two choices of generic datasets one is MNLI and SNLI, SNLI statemetns were short, and image captions and large dataset(550k) whereas MNLI have long sentences, medium dataset(330k).
In future we will try to merge datasets.

In [ ]:
# Standardize columns


# Format Class 1 (Norms)
ds_cb = ds_cb.rename_column("eval_whole_desc", "text")
ds_cb = ds_cb.select_columns(["text"])
ds_cb = ds_cb.add_column("label", [1] * len(ds_cb))

# Format Class 0 (Generic)
ds_snli = ds_snli.rename_column("premise", "text")
ds_snli = ds_snli.select_columns(["text"])
ds_snli = ds_snli.add_column("label", [0] * len(ds_snli))




In [ ]:
#showing that snli contains duplicates

import pandas as pd

##showing that snli contains duplicates.
snli_df = ds_snli.to_pandas()
# Sort by premise length (or any column you want)
snli_df_sorted = snli_df.sort_values("text").reset_index(drop=True)
# Print first 100 rows
print(snli_df_sorted.head(100).to_string())




In [ ]:
# deduplicating snli using pandas and balancing the dataset.
snli_df = snli_df.drop_duplicates(subset=["text"])

# Convert back to a Hugging Face dataset if needed
from datasets import Dataset

# ADDED 'preserve_index=False' HERE TO PREVENT THE EXTRA COLUMN
ds_snli = Dataset.from_pandas(snli_df, preserve_index=False)

print(f"SNLI final size: {len(ds_snli)}")

# Downsample SNLI to match CultureBank exactly
ds_snli = ds_snli.shuffle(seed=42).select(range(len(ds_cb)))

print("Both datasets now have 'text' and 'label' columns.")
print(f"Class 1 (Norms) size: {len(ds_cb)}")
print(f"Class 0 (Generic) size: {len(ds_snli)}")



In [ ]:
# Inspect the datasets before merging


print("--- Class 1: Cultural Norms (CultureBank) ---")
# First 5 rows for checking datasets
display(ds_cb.select(range(5)).to_pandas())

print("\n--- Class 0: Generic Sentences (SNLI) ---")
display(ds_snli.select(range(5)).to_pandas())

In [ ]:
# Merge datasets and clean empty rows

ds_combined = concatenate_datasets([ds_cb, ds_snli])

# Shuffle to mix the generic sentences and norms (for fun??)
ds_combined = ds_combined.shuffle(seed=42)

# Remove any empty or completely whitespace rows
ds_combined = ds_combined.filter(lambda x: x["text"] is not None and len(x["text"].strip()) > 0)

print(f"Total unified dataset size: {len(ds_combined)} rows\n")

print("--- Sneak Peek at the Merged & Shuffled Data ---")
# Display the first 10 rows to prove the 1s and 0s are properly mixed!
display(ds_combined.select(range(10)).to_pandas())


In [ ]:
# Split into Train (80%), Val (10%), Test (10%)
from datasets import ClassLabel

# Hugging Face requires the label column to be a 'ClassLabel' type to use stratify_by_column
ds_combined = ds_combined.cast_column("label", ClassLabel(num_classes=2, names=["generic", "norm"]))

print("Splitting data...")

# First split: 80% Train, 20% Temp
split_1 = ds_combined.train_test_split(test_size=0.20, seed=42, stratify_by_column="label")
train_ds = split_1["train"]
temp_ds = split_1["test"]

# Second split: Cut the 20% Temp into 10% Val and 10% Test
split_2 = temp_ds.train_test_split(test_size=0.50, seed=42, stratify_by_column="label")
val_ds = split_2["train"]
test_ds = split_2["test"]

print(f"Train size: {len(train_ds)}")
print(f"Validation size: {len(val_ds)}")
print(f"Test size: {len(test_ds)}")

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

print("Downloading Tokenizers...")
# 1. Get the Tokenizers
tokenizer_bert = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer_roberta = AutoTokenizer.from_pretrained("roberta-base")

# 2. Define the separate tokenization functions
def tokenize_bert(examples):
    return tokenizer_bert(examples["text"], padding="max_length", truncation=True, max_length=128)

def tokenize_roberta(examples):
    return tokenizer_roberta(examples["text"], padding="max_length", truncation=True, max_length=128)

print("Applying tokenizers to data (Train, Val, and Test)...")
# 3. Process the Data (Added test_ds here)
train_bert = train_ds.map(tokenize_bert, batched=True)
val_bert = val_ds.map(tokenize_bert, batched=True)
test_bert = test_ds.map(tokenize_bert, batched=True)

train_roberta = train_ds.map(tokenize_roberta, batched=True)
val_roberta = val_ds.map(tokenize_roberta, batched=True)
test_roberta = test_ds.map(tokenize_roberta, batched=True)

# 4. Format for PyTorch
cols = ["input_ids", "attention_mask", "label"]
# Included the test datasets in the formatting loop
for ds in [train_bert, val_bert, test_bert, train_roberta, val_roberta, test_roberta]:
    ds.set_format("torch", columns=cols)

# 5. Create the DataLoaders
BATCH_SIZE = 16

# Loaders for BERT
loader_train_bert = DataLoader(train_bert, batch_size=BATCH_SIZE, shuffle=True)
loader_val_bert = DataLoader(val_bert, batch_size=BATCH_SIZE)
loader_test_bert = DataLoader(test_bert, batch_size=BATCH_SIZE)

# Loaders for RoBERTa
loader_train_roberta = DataLoader(train_roberta, batch_size=BATCH_SIZE, shuffle=True)
loader_val_roberta = DataLoader(val_roberta, batch_size=BATCH_SIZE)
loader_test_roberta = DataLoader(test_roberta, batch_size=BATCH_SIZE)

print("All Tokenizers and DataLoaders (including Test) are ready!")

In [ ]:
#zero shot model



import torch
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Load the Pretrained Model
# This loads the base BERT knowledge but attaches a fresh, untrained classifier on top
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).to(device)
model.eval()

all_preds = []
all_labels = []

print("Running inference on test set (Zero-Shot/Baseline)...")
with torch.no_grad():
    for batch in loader_test_bert:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)

        # Get the index of the highest logit (0 for Generic, 1 for Norm)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# 3. Results and Analysis
print("\n" + "="*30)
print("  BASELINE EVALUATION REPORT  ")
print("="*30)
print(classification_report(all_labels, all_preds, target_names=["Generic", "Norm"]))

# 4. Plotting the Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Generic", "Norm"])
disp.plot(cmap=plt.cm.Greens)
plt.title("Confusion Matrix: BERT Baseline (Pre-Finetuning)")
plt.show()

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import matplotlib.pyplot as plt
import numpy as np

!pip install evaluate
import evaluate

# 1. Define the Metric and Function (Crucial to keep this in the same block)
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 2. Set your model name (Change this to switch models)
model_name = "distilbert-base-uncased"
#bert-base-uncased or roberta-base-uncased or distilbert-base-uncased


# 3. Initialize the Universal Model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 4. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch", # Changed from evaluation_strategy to eval_strategy
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
    report_to="none"
)

# 5. Select the correct tokenized data
if "roberta" in model_name:
    train_data, val_data, test_data = train_roberta, val_roberta, test_roberta
else:
    # Use BERT/DistilBERT tokenized data
    train_data, val_data, test_data = train_bert, val_bert, test_bert

# 6. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    compute_metrics=compute_metrics,
)

# 7. Execute Training
print(f"Starting Fine-Tuning for {model_name}...")
trainer.train()

In [ ]:
# Extract logs from the trainer
history = trainer.state.log_history

# Separate training and evaluation logs
train_loss = [x['loss'] for x in history if 'loss' in x]
train_steps = [x['step'] for x in history if 'loss' in x]

val_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]
val_steps = [x['step'] for x in history if 'eval_loss' in x]

# Create the plot
plt.figure(figsize=(10, 6))

# Plot training loss (logged every 10 steps)
plt.plot(train_steps, train_loss, label='Training Loss', color='deepskyblue', alpha=0.6, linestyle='--')

# Plot validation loss (logged every epoch)
plt.plot(val_steps, val_loss, label='Validation Loss', color='red', marker='o', linewidth=2)

plt.title(f'Training vs Validation Loss: {model_name}')
plt.xlabel('Training Steps')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import torch

# 1. Run PREDICT on test set
# This automatically handles model.eval() and torch.no_grad()
print(f"Generating Predictions on Test Set using {model_name}...")
test_results = trainer.predict(test_data)

# 2. Extract Metrics and Predictions
# test_results.metrics contains F1, loss, etc.
# test_results.predictions contains the raw logits
print("\n--- Final Test Metrics ---")
for key, value in test_results.metrics.items():
    print(f"{key}: {value:.4f}")

# 3. Process Predictions for Analysis
y_pred = np.argmax(test_results.predictions, axis=-1)
y_true = test_results.label_ids

# 4. Print Classification Report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Generic", "Norm"]))

# 5. Plot Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Generic", "Norm"])
disp.plot(cmap=plt.cm.Blues)
plt.title(f"Confusion Matrix: {model_name}")
plt.show()

In [ ]:
import torch.nn.functional as F

def predict_custom_sentence(sentence, model, tokenizer):
    # 1. Prepare the model and input
    model.eval()
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True, max_length=128)

    # Move inputs to the same device as the model (GPU or CPU)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # 2. Get the prediction
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)
        prediction = torch.argmax(probs, dim=-1).item()

    # 3. Map the result back to the labels
    label_map = {0: "Generic", 1: "Cultural Norm"}
    result = label_map[prediction]
    confidence = probs[0][prediction].item()

    print(f"Sentence: '{sentence}'")
    print(f"Prediction: {result} ({confidence*100:.2f}% confidence)")
    print("-" * 30)

# --- TEST IT OUT ---
# Make sure to use tokenizer_bert if you used DistilBERT/BERT
my_sentence = "You should boil water before serving it to guests in this region."
predict_custom_sentence(my_sentence, model, tokenizer_bert)

# Try another one just to see the difference
generic_sentence = "the sky is blue today"
predict_custom_sentence(generic_sentence, model, tokenizer_bert)

Using `AutoModelForSequenceClassification` is the most efficient way to implement this. It automatically handles the architectural "plumbing"—attaching a randomly initialized classification head (a linear layer) to the pre-trained BERT backbone and ensuring the output of the `[CLS]` token is correctly routed.

Here is the step-by-step breakdown of the process using the Hugging Face ecosystem.

---

## 1. Environment Setup & Model Loading
The `Auto` classes are designed to be model-agnostic. If you decide to switch from `bert-base-uncased` to a multilingual model like `xlm-roberta-base` later, you only need to change the string name.

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "bert-base-uncased"

# Load Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
```

---

## 2. Preprocessing: The "Norm" Data Pipeline
To classify text, your input must be converted into a format the model understands.

* **Padding:** Ensures all sequences in a batch are the same length.
* **Truncation:** Cuts off text that exceeds BERT's 512-token limit.
* **Return Tensors:** Specifies `pt` for PyTorch.

```python
texts = ["You should always greet elders first.", "The sun rises in the east."]
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

# inputs contains:
# 1. input_ids (token indices)
# 2. attention_mask (1 for real tokens, 0 for padding)
# 3. token_type_ids (segment markers)
```

---

## 3. The Forward Pass (Model Architecture)
When you call `model(**inputs)`, the following operations occur inside the `AutoModelForSequenceClassification` wrapper:



1.  **Encoder Pass:** The tokens pass through 12 layers of Transformer blocks.
2.  **Pooling:** The model extracts the hidden state of the **[CLS] token** from the final layer.
3.  **Dropout:** A dropout layer is applied to reduce overfitting to specific "norm" keywords.
4.  **Classification Head:** A linear layer ($W^T x + b$) transforms the 768-dimensional vector into a 2-dimensional vector (Logits).

---

## 4. Model Training (Fine-Tuning)
For a data scientist, the `Trainer` API is the cleanest way to handle the training loop, evaluation, and checkpointing.

### Define Training Arguments
You'll want to focus on a low learning rate to preserve BERT's linguistic knowledge.

```python
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5, # Critical for BERT stability
    weight_decay=0.01,
    logging_dir="./logs",
    evaluation_strategy="epoch"
)
```

### The Trainer Initialization
The `Trainer` handles the backpropagation and loss calculation (Cross-Entropy) automatically.

```python
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
)

trainer.train()
```

---

## 5. Inference: Norm vs. General
Once trained, the model outputs **Logits** (raw scores). You must apply a **Softmax** to turn these into probabilities.

```python
outputs = model(**inputs)
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)

# Example Output: [0.98, 0.02] -> Highly likely to be a Norm (Class 0)
```

### Key Considerations for Norm Classification
* **Imbalanced Classes:** If your dataset has 1,000 general statements but only 50 norms, BERT will struggle. Use the `compute_loss` override in the Trainer to implement **Class Weights**.
* **Interpretability:** Since you have a background in **SHAP**, you can pass this `AutoModel` into a SHAP `Explainer` to visualize if the model is focusing on "Deontic" markers (must, should, forbidden) or just random nouns.

Do you currently have a labeled dataset of norms and general statements, or are you planning to use a semi-supervised approach to label your data first?